# Jupyter, conda и научный Python: NumPy и Matplotlib

## Мотивация

Почти вся исследовательская работа в ML начинается в **ноутбуке**: там удобно по шагам проверять идеи и сразу видеть числа и графики рядом с кодом. Чтобы это работало предсказуемо на любой машине — от ноутбука до сервера и Google Colab — код запускают в изолированном **окружении** (`conda`), а тяжёлые вычисления доверяют не циклам Python, а векторизованному **NumPy**. Итог почти всегда нужно показать глазами — это делает **Matplotlib**.

Сегодня соберём этот ежедневный набор: интерактивный ноутбук и его магии → изолированное окружение → быстрые массивы → графики. На нём держатся все дальнейшие семинары.

## 1. Jupyter Notebook: как он устроен

Ноутбук — это последовательность **ячеек**: с кодом или с текстом (Markdown). Код исполняет **ядро** (kernel) — отдельный процесс Python, который хранит состояние между ячейками: переменная, созданная в одной ячейке, видна в следующих. Число `[N]` слева от ячейки — порядковый номер её запуска, а не позиция в ноутбуке.

- `Shift+Enter` — выполнить ячейку и перейти к следующей.
- `Tab` — автодополнение, `Shift+Tab` — подсказка по сигнатуре.
- Kernel → Restart & Run All — перезапустить ядро и выполнить всё сверху вниз.

### Хорошие практики

В конце работы перезапустите ядро и выполните все ячейки сверху вниз (Kernel → Restart & Run All) — и убедитесь, что ноутбук отрабатывает **без ошибок**. Так проверяется воспроизводимость: итог не должен зависеть от того, в каком порядке вы запускали ячейки во время работы.

### Лучшие практики

Ноутбук удобен для обучения и разведки: запустил маленький кусочек кода — сразу увидел результат. Но когда кода становится много, поддерживать его в ноутбуке практически невозможно — теряются структура, переиспользование и тестируемость. Поэтому «боевой» код принято держать в репозитории: модулями `.py` под контролем версий (а всё чаще — редактируя их кодовыми агентами). Практическое правило: прототип и разведка — в ноутбуке, а устоявшуюся логику переносите в модули репозитория.

In [ ]:
answer = 42                 # переменная останется в памяти ядра
greeting = "hello, kernel"
answer * 2                  # значение последнего выражения печатается без print

#### ❓ **Вопрос**: Почему один и тот же ноутбук может дать разный результат при запуске ячеек в разном порядке?

<details>

<summary><strong>Ответ</strong></summary>

Ядро хранит состояние (значения переменных) и меняет его в том порядке, в котором вы запускаете ячейки, а не в котором они расположены. Если выполнить ячейки не по порядку или несколько раз, значение переменной может разойтись с кодом выше. Надёжная проверка воспроизводимости — Restart & Run All: перезапуск ядра и выполнение всех ячеек сверху вниз.

</details>

## 2. Магии ноутбука: `%` и `%%`

**Магии** — специальные команды IPython-ядра, которых нет в обычном Python:

- **Строчная магия** `%` действует на одну строку: `%timeit`, `%pwd`, `%who`, `%run script.py`.
- **Ячейковая магия** `%%` действует на **всю ячейку** и стоит первой строкой: `%%time`, `%%bash`, `%%writefile file.py`.
- `!команда` выполняет команду shell: `!pip install numpy`, `!nvidia-smi`.

Частые магии: `%timeit` — усреднённый замер времени выражения; `%%time` — время всей ячейки; `%who` / `%whos` — список определённых переменных.

Раньше вывод графиков Matplotlib «включали» магией `%matplotlib inline`. Сейчас она почти везде не нужна — в свежих Jupyter и в Google Colab отрисовка графиков в ноутбук включена по умолчанию.

In [ ]:
!python --version           # ! — команда shell прямо из ноутбука
%who                        # какие переменные уже определены в ядре
%timeit sum(range(100_000)) # усреднённый замер времени выражения

#### ❓ **Вопрос**: Чем строчная магия `%` отличается от ячейковой `%%`?

<details>

<summary><strong>Ответ</strong></summary>

`%` — строчная магия: относится к одной строке (`%timeit выражение`). `%%` — ячейковая: должна стоять первой строкой и относится ко всей ячейке целиком (`%%time`, `%%bash`, `%%writefile`). Поэтому `%timeit` меряет одно выражение, а `%%time` — весь код ячейки.

</details>

### Интерактивные графики: plotly

Ноутбук умеет показывать не только статичные картинки, но и **интерактивный** вывод. Библиотека `plotly` строит графики, которые можно зумить, вращать и наводить курсор на точки прямо в ячейке — в Colab это работает из коробки. Установка: `!pip install plotly`.

In [ ]:
import plotly.express as px

df = px.data.iris()                        # встроенный набор данных (таблица pandas)
fig = px.scatter(
    df,
    x="sepal_width",
    y="sepal_length",
    color="species",
    size="petal_length",
    hover_data=["petal_width"],
    title="Ирисы Фишера — наведи курсор на точку",
)
fig.show()

## 3. conda: изолированное окружение

`conda` — менеджер пакетов и **окружений**. Окружение — это отдельный каталог со своим Python и своим набором библиотек. Разным проектам нужны разные версии (`numpy 1.x` против `2.x`), а установка всего в системный Python рано или поздно ломает зависимости. Изолированное окружение решает это: сломал — удалил и создал заново, не задев остальное.

Базовые команды (выполняются в терминале):

```bash
conda create -n ml-course python=3.11    # создать окружение
conda activate ml-course                  # войти в него
conda install numpy matplotlib            # поставить пакеты
conda env list                            # список окружений
conda env export > environment.yml        # зафиксировать состав
conda env create -f environment.yml       # воспроизвести из файла
```

In [ ]:
%%bash
conda --version
which python
python --version
conda env list | head -n 5

#### ❓ **Вопрос**: Зачем создавать отдельное окружение, а не ставить пакеты в системный Python?

<details>

<summary><strong>Ответ</strong></summary>

Разным проектам нужны несовместимые версии одних и тех же библиотек, а установка всего в один системный Python приводит к конфликтам зависимостей и может сломать инструменты ОС. Отдельное окружение изолирует набор пакетов: его легко воспроизвести из `environment.yml` и безболезненно удалить и пересоздать.

</details>

## 4. NumPy: массивы и векторизация

`ndarray` — массив чисел одного типа, лежащих в памяти подряд. Ключевые атрибуты — `shape` (форма) и `dtype` (тип элементов); это **атрибуты, а не методы**, пишутся без скобок. Массивы создают через `np.array`, `np.zeros`, `np.ones`, `np.arange`, `np.linspace`.

Операции применяются **поэлементно, без циклов** (векторизация): `a ** 2`, `np.sqrt(a)`, `a + b`. **Broadcasting** «растягивает» массивы совместимых форм: `a + 10` прибавляет скаляр к каждому элементу. Векторизованный код короче и в десятки раз быстрее цикла Python, потому что операция выполняется единым проходом внутри скомпилированного кода над непрерывной памятью.

In [ ]:
import numpy as np

a = np.array([1, 4, 9, 16, 25])
print("shape:", a.shape, "| dtype:", a.dtype)   # атрибуты, без ()
print("a ** 2  =", a ** 2)                       # поэлементно
print("sqrt(a) =", np.sqrt(a))                   # np.sqrt — корень (np.square — квадрат!)
print("a + 10  =", a + 10)                       # broadcasting: скаляр к каждому

x = np.linspace(0, 1, 5)                          # 5 точек от 0 до 1 включительно
print("linspace:", x)

#### ❓ **Вопрос**: Почему `list(range(5)) + [10]` и `np.arange(5) + 10` дают разное?

<details>

<summary><strong>Ответ</strong></summary>

Для списка Python оператор `+` — это **конкатенация**: `[0,1,2,3,4] + [10]` даёт `[0,1,2,3,4,10]`. Для массива NumPy `+` — **поэлементная** операция, и скаляр `10` по правилам broadcasting прибавляется к каждому элементу: `[10,11,12,13,14]`.

</details>

In [ ]:
big = np.arange(1_000_000)

%timeit sum(i * i for i in range(1_000_000))    # чистый Python — медленно
%timeit (big * big).sum()                        # векторизованный NumPy — быстро

#### ❓ **Вопрос**: Почему `(big * big).sum()` в разы быстрее цикла Python по тем же числам?

<details>

<summary><strong>Ответ</strong></summary>

В цикле Python на каждый элемент создаётся объект и выполняется интерпретируемый код. NumPy хранит числа одного типа подряд в памяти и выполняет операцию единым проходом внутри скомпилированного кода (C), без пооэлементных Python-объектов и накладных расходов интерпретатора.

</details>

## 5. Matplotlib: первый график

`matplotlib.pyplot` строит графики. Быстрый способ — вызвать `plt.plot(x, y)`, затем добавить подписи и показать. Для нескольких графиков на фигуре удобен объектный интерфейс: `fig, ax = plt.subplots()`.

Правила читаемого графика: подписать оси (`xlabel`, `ylabel`), дать заголовок (`title`), включить сетку (`grid`), а при нескольких кривых — легенду (`legend`; для неё у `plot` нужен `label=`). Важно: `savefig` вызывают **до** `show()` — `show()` очищает фигуру, и после него файл получится пустым.

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 200)          # много точек → гладкая кривая
plt.plot(x, np.sin(x), label="sin(x)")
plt.plot(x, np.cos(x), label="cos(x)")
plt.xlabel("x")
plt.ylabel("y")
plt.title("sin и cos на [0, 2π]")
plt.legend()
plt.grid(True)
plt.savefig("sin_cos.png", dpi=150)          # СНАЧАЛА сохранить...
plt.show()                                   # ...потом показать

#### ❓ **Вопрос**: Почему `plt.savefig(...)` нужно вызывать до `plt.show()`?

<details>

<summary><strong>Ответ</strong></summary>

`plt.show()` отрисовывает и **очищает** текущую фигуру. Если вызвать `savefig` после `show()`, сохранять будет уже нечего — файл выйдет пустым. Поэтому порядок: сначала `savefig`, затем `show`.

</details>

### Разные типы графиков и оформление

Matplotlib умеет много типов графиков и тонкую настройку внешнего вида:

- **тип графика**: `plot` (линия), `scatter` (точки), `bar` (столбцы), `hist` (гистограмма), `fill_between` (заливка области);
- **стиль линии** `linestyle`: `"-"` сплошная, `"--"` пунктир (dashed), `":"` точки (dotted), `"-."` штрихпунктир (dash-dot);
- **цвет** `color` (`"crimson"`, `"#1f77b4"`), **толщина** `linewidth`, **маркеры** `marker="o"`;
- **полупрозрачность** `alpha` от 0 до 1 — спасает, когда линии или точки накладываются друг на друга.

In [ ]:
x = np.linspace(0, 2 * np.pi, 200)

plt.figure(figsize=(9, 4))
plt.plot(x, np.sin(x),       color="crimson",   linestyle="-",  linewidth=2, label="сплошная (solid)")
plt.plot(x, np.sin(x - 0.6), color="steelblue", linestyle="--", linewidth=2, label="пунктир (dashed)")
plt.plot(x, np.sin(x - 1.2), color="green",     linestyle=":",  linewidth=2, label="точки (dotted)")
plt.plot(x, np.sin(x - 1.8), color="purple",    linestyle="-.", linewidth=2, label="штрихпунктир (dash-dot)")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Стили линий, цвета и толщина")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(size=400)
y = 0.6 * x + rng.normal(size=400) * 0.4

plt.figure(figsize=(6, 5))
sc = plt.scatter(x, y, c=y, cmap="viridis", alpha=0.6, edgecolors="k", linewidths=0.3)
plt.colorbar(sc, label="значение y")
plt.xlabel("x")
plt.ylabel("y")
plt.title("scatter: цвет по значению, полупрозрачность alpha=0.6")
plt.show()

#### ❓ **Вопрос**: Зачем на диаграмме рассеяния полупрозрачность (`alpha`)?

<details>

<summary><strong>Ответ</strong></summary>

Когда точек много и они накладываются, при `alpha=1` всё сливается в сплошное пятно и не видно, где точек больше. Полупрозрачность делает перекрытия темнее, поэтому по насыщенности цвета читается плотность — где данных много, а где мало.

</details>

## 6. Google Colab

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](#)

<!-- TODO: подставить ссылку на ноутбук в Colab в финальной версии — итоговый репозиторий пока не определён. -->

**Google Colab** — это Jupyter в облаке: ноутбук исполняется на удалённой машине Google с бесплатным доступом к GPU/TPU (Runtime → Change runtime type). Основные библиотеки (`numpy`, `matplotlib`, `pandas`, `torch`) уже предустановлены; недостающее ставят через `!pip install ...`. Проверить GPU — `!nvidia-smi`.

**Бесплатный доступ ограничен по времени.** Сессия живёт лишь несколько часов и отключается при простое, а бесплатные GPU не гарантированы и выдаются по остаточному принципу. Для долгих расчётов это неудобно, но для семинаров и небольших экспериментов бесплатного тарифа достаточно.

**В Colab нет conda — только pip.** conda (дистрибутив Anaconda и его основные каналы) — коммерческий продукт: для крупных компаний его использование платное, поэтому Google не встраивает его в Colab, чтобы не платить за лицензию. Для учёбы и личных проектов conda бесплатна — на своей машине пользуйтесь ей спокойно, а в Colab ставьте пакеты через `!pip install`.

Файловая система сессии Colab **временная**: после отключения среды загруженные файлы пропадают. Чтобы сохранить результаты, монтируют Google Drive:

```python
from google.colab import drive
drive.mount('/content/drive')      # файлы появятся в /content/drive/MyDrive
```

#### ❓ **Вопрос**: Куда сохранять результаты в Colab, чтобы они не пропали после отключения среды?

<details>

<summary><strong>Ответ</strong></summary>

Локальная файловая система сессии Colab временная и очищается при остановке среды. Чтобы сохранить графики, модели и данные, монтируют Google Drive (`drive.mount('/content/drive')` из модуля `google.colab`) и пишут в `/content/drive/MyDrive/...`, либо скачивают файлы на свой компьютер.

</details>